# Pertemuan 9 — Greedy II (Graf) dan Pencarian Heuristik
Kruskal, Prim, Dijkstra, Huffman, dan **[Modern]** A* (heuristik admissible/consistent) vs Dijkstra.

In [ ]:
import heapq, random, math, time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
random.seed(9)

## 1. MST: Kruskal (Union-Find) vs Prim (heap)
Keduanya O(E log V); Kruskal bergantung pada sort + union-find (≈ α(V) per operasi), Prim pada priority queue.

In [ ]:
class UF:
    def __init__(self, n): self.p, self.r = list(range(n)), [0] * n
    def find(self, x):
        while self.p[x] != x: self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b: return False
        if self.r[a] < self.r[b]: a, b = b, a
        self.p[b] = a; self.r[a] += self.r[b] == self.r[a]; return True

def kruskal(n, edges):
    uf, tot = UF(n), 0
    for w, u, v in sorted(edges):
        if uf.union(u, v): tot += w
    return tot

def prim(n, adj):
    seen, pq, tot = [False] * n, [(0, 0)], 0
    while pq:
        w, u = heapq.heappop(pq)
        if seen[u]: continue
        seen[u] = True; tot += w
        for v, wt in adj[u]:
            if not seen[v]: heapq.heappush(pq, (wt, v))
    return tot

def random_graph(n, m):
    edges = {(i, i + 1): random.random() for i in range(n - 1)}          # pastikan terhubung
    while len(edges) < m:
        u, v = random.sample(range(n), 2); edges[(min(u, v), max(u, v))] = random.random()
    adj = [[] for _ in range(n)]
    for (u, v), w in edges.items(): adj[u].append((v, w)); adj[v].append((u, w))
    return [(w, u, v) for (u, v), w in edges.items()], adj

print(f"{'V':>6} {'E':>7} {'Kruskal(s)':>11} {'Prim(s)':>9} sama?")
for n, m in [(1000, 5000), (5000, 25000), (20000, 100000)]:
    E, adj = random_graph(n, m)
    t = time.perf_counter(); a = kruskal(n, E); ta = time.perf_counter() - t
    t = time.perf_counter(); b = prim(n, adj);  tb = time.perf_counter() - t
    print(f"{n:>6} {m:>7} {ta:>11.4f} {tb:>9.4f} {abs(a-b) < 1e-9}")

## 2. Dijkstra

In [ ]:
def dijkstra(adj, s):
    dist = [math.inf] * len(adj); dist[s] = 0; pq = [(0, s)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        for v, w in adj[u]:
            if d + w < dist[v]: dist[v] = d + w; heapq.heappush(pq, (dist[v], v))
    return dist
E, adj = random_graph(2000, 10000)
d = dijkstra(adj, 0); print("jarak maksimum dari simpul 0:", round(max(d), 3), "| semua terjangkau:", all(x < math.inf for x in d))

## 3. Huffman Coding
Rata-rata panjang kode harus ≥ entropi H dan < H+1.

In [ ]:
def huffman(text):
    freq = Counter(text); heap = [[w, i, [c, ""]] for i, (c, w) in enumerate(freq.items())]; heapq.heapify(heap); i = len(heap)
    while len(heap) > 1:
        lo, hi = heapq.heappop(heap), heapq.heappop(heap)
        for p in lo[2:]: p[1] = "0" + p[1]
        for p in hi[2:]: p[1] = "1" + p[1]
        heapq.heappush(heap, [lo[0] + hi[0], i] + lo[2:] + hi[2:]); i += 1
    return {c: code for c, code in heap[0][2:]}, freq
text = "algoritma kompleksitas strategi algoritma greedy huffman coding optimal " * 20
codes, freq = huffman(text); N = len(text)
avg = sum(freq[c] * len(codes[c]) for c in freq) / N
H = -sum(f / N * math.log2(f / N) for f in freq.values())
print(f"panjang kode rata-rata = {avg:.3f} bit | entropi H = {H:.3f} | ASCII = 8 bit | rasio = {avg/8:.2%}")

## 4. [Modern] A* vs Dijkstra pada grid
Dijkstra = A* dengan h ≡ 0. Heuristik Manhattan *admissible* dan *consistent* untuk grid 4-arah berbiaya 1 → A* tetap optimal tetapi memperluas jauh lebih sedikit simpul.

In [ ]:
def make_grid(n=60, p=0.25, seed=0):
    r = random.Random(seed)
    while True:
        g = [[r.random() < p for _ in range(n)] for _ in range(n)]
        g[0][0] = g[n-1][n-1] = False
        if astar(g, (0, 0), (n-1, n-1), lambda a, b: 0)[0] is not None: return g

def astar(g, s, goal, h, w=1.0):
    n = len(g); gscore = {s: 0}; closed = set(); pq = [(w * h(s, goal), 0, s)]; expanded = 0
    while pq:
        f, gc, u = heapq.heappop(pq)
        if u in closed: continue
        closed.add(u); expanded += 1
        if u == goal: return gc, expanded
        x, y = u
        for v in ((x+1, y), (x-1, y), (x, y+1), (x, y-1)):
            if 0 <= v[0] < n and 0 <= v[1] < n and not g[v[0]][v[1]] and gc + 1 < gscore.get(v, math.inf):
                gscore[v] = gc + 1; heapq.heappush(pq, (gc + 1 + w * h(v, goal), gc + 1, v))
    return None, expanded

manhattan = lambda a, b: abs(a[0] - b[0]) + abs(a[1] - b[1])
zero = lambda a, b: 0
print(f"{'seed':>4} | {'Dijkstra (panjang, expanded)':>30} | {'A* Manhattan':>18} | {'A* berbobot w=3 (tidak admissible)':>36}")
for seed in range(5):
    g = make_grid(seed=seed)
    r0, r1, r2 = (astar(g, (0, 0), (59, 59), h, w) for h, w in ((zero, 1), (manhattan, 1), (manhattan, 3)))
    print(f"{seed:>4} | {str(r0):>30} | {str(r1):>18} | {str(r2):>36}")

**Amati:** panjang jalur A* (w=1) = Dijkstra; w=3 memperluas lebih sedikit simpul tetapi bisa menghasilkan jalur lebih panjang (trade-off optimalitas vs kecepatan).

## 5. Latihan / Tugas
1. Tambahkan gerakan diagonal (biaya √2) — heuristik apa yang masih admissible? (Euclidean / Chebyshev/octile)
2. Plot jumlah simpul diperluas vs ukuran grid n untuk Dijkstra dan A*.
3. **Tugas:** bandingkan Kruskal vs Prim pada graf padat (E ≈ V²/4) dan jarang (E ≈ 2V). Mana yang lebih cepat dan mengapa?
4. **Tugas:** Dijkstra vs A* pada beberapa pola rintangan (acak, labirin, dinding tengah) — kapan A* tidak banyak membantu?

In [ ]:
def octile(a, b):
    pass  # TODO